# Notebook 1 — Confirm Domain Distribution in `yelp_sampled_100k.csv`

This notebook answers one question: **what domains/categories are actually present** in the sampled Yelp CSV, and how are they distributed?  
Run this before the extraction notebook to know exactly which domains are thin and need boosting.

**Drive path expected:** `MyDrive/NaijaReview_Data/yelp/yelp_sampled_100k.csv`

In [ ]:
# ── Cell 1: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 2: Load CSV and inspect columns ────────────────────────────────────
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/NaijaReview_Data/yelp/yelp_sampled_100k.csv"

print("Loading CSV (this may take ~30s for 100k rows)...")
df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst row sample:")
print(df.iloc[0].to_dict())

In [ ]:
# ── Cell 3: Check for category / domain columns ──────────────────────────────
# The CSV may have come from the Yelp business join, so we check what's available.
category_cols = [c for c in df.columns if any(k in c.lower() for k in ['categ', 'domain', 'type', 'genre'])]
print("Candidate category columns:", category_cols)

# Also show null counts for all columns
print("\nNull counts per column:")
print(df.isnull().sum().to_string())

In [ ]:
# ── Cell 4: Domain distribution (use whichever column exists) ────────────────
# The existing pipeline uses a 'domain' column in final_dataset.jsonl.
# This cell checks whether the CSV already has that, or if we need to derive it.

from collections import Counter

DOMAIN_COL = None
for candidate in ['domain', 'category', 'categories', 'yelp_category', 'business_category']:
    if candidate in df.columns:
        DOMAIN_COL = candidate
        break

if DOMAIN_COL:
    print(f"Using column: '{DOMAIN_COL}'")
    dist = df[DOMAIN_COL].value_counts()
    print(f"\nTop 40 values:")
    print(dist.head(40).to_string())
    print(f"\nUnique values: {df[DOMAIN_COL].nunique()}")
    print(f"Null / missing: {df[DOMAIN_COL].isnull().sum()}")
else:
    print("No obvious category/domain column found.")
    print("Will need to derive from Yelp category strings in the next step.")

In [ ]:
# ── Cell 5: Map raw Yelp categories → NaijaReview domains ───────────────────
# Uses the same domain taxonomy as the existing pipeline.

DOMAIN_KEYWORD_MAP = {
    'food':          ['restaurant', 'food', 'pizza', 'burger', 'sushi', 'bakery',
                      'breakfast', 'brunch', 'seafood', 'steakhouse', 'cafe',
                      'coffee', 'dessert', 'ice cream', 'juice', 'nigerian',
                      'african', 'mexican', 'italian', 'chinese', 'thai', 'deli',
                      'salad', 'chicken', 'sandwich', 'vegan', 'bbq'],
    'hospitality':   ['hotel', 'motel', 'hostel', 'resort', 'bed & breakfast',
                      'vacation rental', 'travel', 'airport', 'guest house'],
    'retail':        ['shopping', 'store', 'shop', 'market', 'bookstore', 'book',
                      'clothing', 'fashion', 'grocery', 'supermarket', 'convenience',
                      'furniture', 'electronics', 'jewel', 'gift', 'flower',
                      'liquor', 'beer', 'wine', 'spirits', 'home decor'],
    'services':      ['plumber', 'electrician', 'contractor', 'cleaning', 'repair',
                      'home service', 'laundry', 'dry clean', 'shipping', 'printing',
                      'notary', 'financial', 'insurance', 'legal', 'lawyer',
                      'accountant', 'tax', 'it service', 'computer', 'locksmith'],
    'health':        ['doctor', 'dentist', 'hospital', 'clinic', 'pharmacy',
                      'medical', 'optometrist', 'chiropractor', 'therapist',
                      'mental health', 'urgent care', 'pediatric', 'dermatolog',
                      'gynecolog', 'orthopedic', 'nursing', 'rehab'],
    'automotive':    ['auto', 'car', 'mechanic', 'tire', 'oil change', 'dealership',
                      'car wash', 'body shop', 'transmission', 'towing',
                      'motorcycle', 'truck', 'vehicle'],
    'personal_care': ['hair salon', 'barber', 'nail salon', 'spa', 'massage',
                      'beauty', 'waxing', 'skin care', 'facial', 'cosmetic',
                      'eyebrow', 'lash', 'tattoo', 'piercing'],
    'entertainment': ['nightlife', 'bar', 'club', 'karaoke', 'bowling', 'cinema',
                      'movie', 'theater', 'theatre', 'museum', 'gallery', 'art',
                      'arcade', 'escape room', 'comedy', 'concert', 'music',
                      'sport', 'fitness', 'gym', 'yoga', 'dance', 'park',
                      'recreation', 'amusement'],
    'education':     ['school', 'college', 'university', 'tutoring', 'education',
                      'training', 'driving school', 'language school', 'daycare',
                      'childcare', 'preschool', 'library'],
}

def map_to_domain(category_str):
    """Map a raw Yelp category string to a NaijaReview domain."""
    if not isinstance(category_str, str):
        return 'general'
    cat_lower = category_str.lower()
    for domain, keywords in DOMAIN_KEYWORD_MAP.items():
        if any(kw in cat_lower for kw in keywords):
            return domain
    return 'general'

# Apply to whatever category column exists, falling back to 'general'
if DOMAIN_COL and DOMAIN_COL != 'domain':
    df['derived_domain'] = df[DOMAIN_COL].apply(map_to_domain)
elif 'domain' in df.columns:
    df['derived_domain'] = df['domain']
else:
    # No category info at all — mark all general
    df['derived_domain'] = 'general'
    print("WARNING: No category column found; all rows marked as 'general'.")

print("Domain distribution after mapping:")
domain_dist = df['derived_domain'].value_counts()
print(domain_dist.to_string())
print(f"\nTotal rows: {len(df):,}")

In [ ]:
# ── Cell 6: Visualise the distribution ──────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
domain_dist.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Review Count by Domain', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Domain')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# Pie chart
domain_dist.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Domain Share (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('/content/domain_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to /content/domain_distribution.png")

In [ ]:
# ── Cell 7: Gap analysis — which domains need extraction ────────────────────
TARGET_PER_DOMAIN = 2000   # minimum we want per domain in the full dataset

print("=" * 55)
print("DOMAIN GAP ANALYSIS")
print("=" * 55)
print(f"{'Domain':<20} {'Count':>8}  {'Status':<20}")
print("-" * 55)

gaps = []
for domain in sorted(DOMAIN_KEYWORD_MAP.keys()) + ['general']:
    count = domain_dist.get(domain, 0)
    status = "OK" if count >= TARGET_PER_DOMAIN else f"NEEDS +{TARGET_PER_DOMAIN - count:,}"
    flag = "✓" if count >= TARGET_PER_DOMAIN else "✗"
    print(f"{flag} {domain:<18} {count:>8,}  {status}")
    if count < TARGET_PER_DOMAIN:
        gaps.append(domain)

print("-" * 55)
print(f"\nDomains needing extraction: {gaps}")
print("\n→ Pass this list to Notebook 2 for targeted extraction.")

In [ ]:
# ── Cell 8: Save the gap report ──────────────────────────────────────────────
import json

report = {
    'total_rows': int(len(df)),
    'domain_counts': {k: int(v) for k, v in domain_dist.items()},
    'domains_needing_extraction': gaps,
    'category_column_used': DOMAIN_COL,
    'csv_columns': list(df.columns)
}

report_path = "/content/drive/MyDrive/NaijaReview_Data/Naiview/data/processed/domain_gap_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"Gap report saved to Drive: {report_path}")
print(json.dumps(report, indent=2))